# Meal Recommendation Model Development

## Overview
This notebook follows a complete machine learning workflow for developing a meal recommendation model for Glocusense diabetes management system.

## Workflow Steps:
1. **Dataset Import** - Load and explore the dataset
2. **Data Cleaning & Preprocessing** - Handle missing values, outliers, data types
3. **Exploratory Data Analysis (EDA)** - Understand data patterns and relationships
4. **Feature Engineering** - Create new features from existing data
5. **Feature Selection** - Select most important features
6. **Data Splitting** - Split into train/validation/test sets
7. **Model Creation** - Build and train multiple models
8. **Model Evaluation** - Compare models and select best one
9. **Save Best Model** - Serialize the best model
10. **Integration** - Prepare model for system integration

---


## Step 1: Import Libraries and Dataset

### 1.1 Import Required Libraries


In [ ]:
# Standard library imports
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Data manipulation and analysis
import pandas as pd
import numpy as np

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import classification_report, confusion_matrix

# Model persistence
import joblib
import pickle

# Database connection (for loading data from Glocusense database)
import sqlite3
from pathlib import Path

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

# Set random seed for reproducibility
np.random.seed(42)

print("[SUCCESS] All libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Scikit-learn version: {__import__('sklearn').__version__}")


### 1.2 Load Dataset

**Options for loading data:**
1. From CSV/Excel file
2. From Glocusense database
3. From external API
4. Generate synthetic data for testing


In [ ]:
# Option 1: Load from CSV file
# dataset_path = 'data/meal_recommendation_dataset.csv'
# df = pd.read_csv(dataset_path)
# print(f"Dataset loaded from CSV: {df.shape}")

# Option 2: Load from Glocusense database
def load_from_database():
    """Load data from Glocusense SQLite database"""
    # Get project root (two levels up from models/)
    project_root = Path(__file__).parent.parent if '__file__' in globals() else Path.cwd().parent
    db_path = project_root / 'instance' / 'glocusense.db'
    
    # Alternative: Check Windows AppData location
    if os.name == 'nt':
        appdata_dir = Path(os.getenv('LOCALAPPDATA', os.path.expanduser('~/.local')))
        db_path = appdata_dir / 'Glocusense' / 'glocusense.db'
    
    if not db_path.exists():
        print(f"[WARNING] Database not found at {db_path}")
        print("Creating sample dataset instead...")
        return create_sample_dataset()
    
    try:
        conn = sqlite3.connect(str(db_path))
        
        # Load food items
        food_items = pd.read_sql_query("SELECT * FROM food_items", conn)
        
        # Load user data
        users = pd.read_sql_query("SELECT * FROM users", conn)
        
        # Load diabetes records
        diabetes_records = pd.read_sql_query("SELECT * FROM diabetes_records", conn)
        
        # Load recommendations (if any)
        recommendations = pd.read_sql_query("SELECT * FROM recommendations", conn)
        
        conn.close()
        
        print(f"[SUCCESS] Loaded {len(food_items)} food items")
        print(f"[SUCCESS] Loaded {len(users)} users")
        print(f"[SUCCESS] Loaded {len(diabetes_records)} diabetes records")
        print(f"[SUCCESS] Loaded {len(recommendations)} recommendations")
        
        return {
            'food_items': food_items,
            'df': food_items,
            'users': users,
            'diabetes_records': diabetes_records,
            'recommendations': recommendations
        }
    except Exception as e:
        print(f"Error loading from database: {e}")
        return create_sample_dataset()

def create_sample_dataset():
    """Create sample dataset from diabetic meal plans CSV or synthetic data"""
    print("Creating sample dataset from diabetic meal plans CSV...")
    for base in [Path.cwd(), Path.cwd().parent]:
        csv_path = base / 'datasets' / 'diabetic_diet_meal_plans_with_macros_GI.csv'
        if csv_path.exists():
            break
    else:
        csv_path = None
    if csv_path and csv_path.exists():
        df_meals = pd.read_csv(csv_path)
        print(f"[SUCCESS] Loaded {len(df_meals)} meal plan rows from CSV")
        return {'food_items': None, 'meal_plans': df_meals, 'df': df_meals}
    df_synth = pd.DataFrame({
        'Dish': ['Posho porridge', 'Beans and rice', 'Chapati with vegetables'],
        'Calories': [336, 350, 400], 'Protein': [19, 15, 18], 'Carbs': [34, 45, 42],
        'Fat': [11, 8, 12], 'Fiber': [6, 8, 7], 'Glycemic Index': [55, 50, 52],
        'Meal': ['Breakfast', 'Lunch', 'Dinner'], 'Group': ['Diabetic_NotActive'] * 3,
    })
    return {'food_items': None, 'meal_plans': df_synth, 'df': df_synth}

# Load data
data = load_from_database()

# Set df for notebook workflow
if data is not None:
    if data.get('food_items') is not None and len(data['food_items']) > 0:
        df = data['food_items']
        print(f"Using food_items from database: {df.shape}")
    elif data.get('df') is not None:
        df = data['df']
        print(f"Using meal plans data: {df.shape}")
    else:
        df = data.get('meal_plans')
        if df is not None:
            print(f"Using meal_plans: {df.shape}")
        else:
            df = None
else:
    df = None


### 1.3 Initial Data Exploration


In [ ]:
# Display basic information about the dataset
if 'df' in locals():
    print("=" * 60)
    print("DATASET OVERVIEW")
    print("=" * 60)
    print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
    print(f"\nColumn Names:")
    print(df.columns.tolist())
    print(f"\nData Types:")
    print(df.dtypes)
    print(f"\nFirst 5 rows:")
    display(df.head())
    print(f"\nLast 5 rows:")
    display(df.tail())
    print(f"\nBasic Statistics:")
    display(df.describe())
    print(f"\nMissing Values:")
    missing = df.isnull().sum()
    print(missing[missing > 0])
else:
    print("[WARNING] Dataset 'df' not loaded. Please load your dataset first.")


## Step 2: Data Cleaning & Preprocessing

### 2.1 Handle Missing Values


In [ ]:
# Check for missing values
if 'df' in locals():
    print("Missing values before cleaning:")
    print(df.isnull().sum())
    
    # Create a copy for cleaning
    df_clean = df.copy()
    
    # Strategy 1: Drop columns with too many missing values (>50%)
    threshold = 0.5
    cols_to_drop = df_clean.columns[df_clean.isnull().mean() > threshold].tolist()
    if cols_to_drop:
        print(f"\nDropping columns with >{threshold*100}% missing values: {cols_to_drop}")
        df_clean = df_clean.drop(columns=cols_to_drop)
    
    # Strategy 2: Fill missing values
    # For numerical columns: use mean, median, or mode
    numerical_cols = df_clean.select_dtypes(include=[np.number]).columns
    for col in numerical_cols:
        if df_clean[col].isnull().sum() > 0:
            # Use median for numerical (more robust to outliers)
            df_clean[col].fillna(df_clean[col].median(), inplace=True)
            print(f"Filled {col} with median: {df_clean[col].median():.2f}")
    
    # For categorical columns: use mode
    categorical_cols = df_clean.select_dtypes(include=['object']).columns
    for col in categorical_cols:
        if df_clean[col].isnull().sum() > 0:
            mode_value = df_clean[col].mode()[0] if not df_clean[col].mode().empty else 'Unknown'
            df_clean[col].fillna(mode_value, inplace=True)
            print(f"Filled {col} with mode: {mode_value}")
    
    print("\nMissing values after cleaning:")
    print(df_clean.isnull().sum().sum(), "missing values remaining")
    
    df = df_clean
else:
    print("[WARNING] Dataset not loaded. Please complete Step 1 first.")


### 2.2 Handle Outliers


In [ ]:
# Detect and handle outliers using IQR method
if 'df' in locals():
    numerical_cols = df.select_dtypes(include=[np.number]).columns
    
    def remove_outliers_iqr(df, column):
        """Remove outliers using IQR method"""
        Q1 = df[column].quantile(0.25)
        Q3 = df[column].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        return df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]
    
    print("Outlier detection and removal:")
    initial_rows = len(df)
    
    for col in numerical_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
        
        if len(outliers) > 0:
            print(f"\n{col}:")
            print(f"  Outliers found: {len(outliers)} ({len(outliers)/len(df)*100:.2f}%)")
            print(f"  Range: [{lower_bound:.2f}, {upper_bound:.2f}]")
            # Option: Remove or cap outliers
            # df = remove_outliers_iqr(df, col)  # Uncomment to remove
            # Or cap outliers:
            df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)
            print(f"  [SUCCESS] Outliers capped")
    
    print(f"\nRows before: {initial_rows}, Rows after: {len(df)}")
else:
    print("[WARNING] Dataset not loaded.")


### 2.3 Data Type Conversions


In [ ]:
# Convert data types appropriately
if 'df' in locals():
    print("Data type conversions:")
    
    # Convert date columns
    date_columns = ['created_at', 'record_date', 'diagnosis_date', 'target_date']
    for col in date_columns:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')
            print(f"[SUCCESS] Converted {col} to datetime")
    
    # Convert boolean columns
    boolean_columns = ['has_diabetes', 'diabetes_friendly', 'profile_completed']
    for col in boolean_columns:
        if col in df.columns:
            df[col] = df[col].astype(bool)
            print(f"[SUCCESS] Converted {col} to boolean")
    
    # Ensure numerical columns are numeric
    numerical_cols = ['age', 'height', 'weight', 'calories', 'protein', 'carbohydrates', 
                      'fiber', 'fat', 'sugar', 'glycemic_index', 'before_breakfast', 
                      'after_breakfast', 'before_lunch']
    for col in numerical_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            print(f"[SUCCESS] Converted {col} to numeric")
    
    print(f"\nFinal data types:")
    print(df.dtypes)
else:
    print("[WARNING] Dataset not loaded.")


## Step 4: Feature Engineering

Create features for meal recommendation: diabetes-friendly flag, meal score, etc.

In [ ]:
# Feature engineering for meal recommendation
if 'df' in locals() and df is not None:
    df_fe = df.copy()
    # Normalize column names (meal plans CSV uses "Glycemic Index")
    gi_col = 'glycemic_index' if 'glycemic_index' in df_fe.columns else 'Glycemic Index'
    if gi_col in df_fe.columns:
        df_fe['diabetes_friendly'] = (df_fe[gi_col] <= 55).astype(int)
        df_fe['meal_score'] = df_fe.apply(lambda r: 100 - (r.get(gi_col, 70) or 70) + (r.get('Fiber', r.get('fiber', 0)) or 0) * 2, axis=1)
    if 'Meal' in df_fe.columns:
        df_fe['meal_type_encoded'] = pd.Categorical(df_fe['Meal']).codes
    df = df_fe
    print("[SUCCESS] Feature engineering complete")

## Step 5 & 6: Feature Selection and Data Splitting

Select features and split for training/validation.

In [ ]:
# Feature selection and split
if 'df' in locals() and df is not None and len(df) > 10:
    num_cols = ['Calories', 'Protein', 'Carbs', 'Fat', 'Fiber', 'Glycemic Index']
    num_cols = [c for c in num_cols if c in df.columns]
    target_col = 'diabetes_friendly' if 'diabetes_friendly' in df.columns else None
    if target_col and num_cols:
        X = df[num_cols].fillna(df[num_cols].median())
        y = df[target_col]
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        print(f"[SUCCESS] Train: {len(X_train)}, Test: {len(X_test)}")
        model_ready = True
    else:
        model_ready = False
else:
    model_ready = False

## Step 7 & 8: Model Creation and Evaluation

Train RandomForest and evaluate.

In [ ]:
# Train and evaluate model
if 'model_ready' in locals() and model_ready:
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    print("Classification Report:")
    print(classification_report(y_test, y_pred))
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Feature importance:", dict(zip(num_cols, model.feature_importances_)))
else:
    print("[INFO] Skipping model: insufficient data or missing target columns")

## Step 9 & 10: Save Model and Integration

Save the best model and scaler for deployment.

In [ ]:
# Save model and scaler
if 'model' in locals() and 'scaler' in locals():
    model_dir = Path(__file__).parent if '__file__' in globals() else Path.cwd()
    model_dir = model_dir / 'saved_models'
    model_dir.mkdir(exist_ok=True)
    joblib.dump(model, model_dir / 'meal_recommendation_model.joblib')
    joblib.dump(scaler, model_dir / 'meal_scaler.joblib')
    joblib.dump(num_cols, model_dir / 'feature_columns.joblib')
    print(f"[SUCCESS] Model saved to {model_dir}")

## Step 3: Exploratory Data Analysis (EDA)

### 3.1 Univariate Analysis


In [ ]:
# Distribution of numerical variables
if 'df' in locals():
    numerical_cols = df.select_dtypes(include=[np.number]).columns
    
    # Create histograms for numerical columns
    n_cols = min(3, len(numerical_cols))
    n_rows = (len(numerical_cols) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows))
    axes = axes.flatten() if len(numerical_cols) > 1 else [axes]
    
    for idx, col in enumerate(numerical_cols[:n_rows*n_cols]):
        df[col].hist(bins=30, ax=axes[idx], edgecolor='black')
        axes[idx].set_title(f'Distribution of {col}')
        axes[idx].set_xlabel(col)
        axes[idx].set_ylabel('Frequency')
    
    plt.tight_layout()
    plt.show()
    
    # Summary statistics
    print("\nSummary Statistics:")
    display(df[numerical_cols].describe())
else:
    print("[WARNING] Dataset not loaded.")


### 3.2 Bivariate Analysis


In [ ]:
# Correlation matrix for numerical variables
if 'df' in locals():
    numerical_cols = df.select_dtypes(include=[np.number]).columns
    
    if len(numerical_cols) > 1:
        # Calculate correlation matrix
        corr_matrix = df[numerical_cols].corr()
        
        # Plot correlation heatmap
        plt.figure(figsize=(12, 10))
        sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
                    center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8})
        plt.title('Correlation Matrix of Numerical Variables')
        plt.tight_layout()
        plt.show()
        
        # Find highly correlated features
        high_corr_pairs = []
        for i in range(len(corr_matrix.columns)):
            for j in range(i+1, len(corr_matrix.columns)):
                if abs(corr_matrix.iloc[i, j]) > 0.7:
                    high_corr_pairs.append((
                        corr_matrix.columns[i], 
                        corr_matrix.columns[j], 
                        corr_matrix.iloc[i, j]
                    ))
        
        if high_corr_pairs:
            print("\nHighly correlated pairs (|r| > 0.7):")
            for col1, col2, corr in high_corr_pairs:
                print(f"  {col1} ↔ {col2}: {corr:.3f}")
        else:
            print("\nNo highly correlated pairs found.")
else:
    print("[WARNING] Dataset not loaded.")


### 3.3 Categorical Variable Analysis


In [ ]:
# Analyze categorical variables
if 'df' in locals():
    categorical_cols = df.select_dtypes(include=['object', 'category']).columns
    
    for col in categorical_cols:
        print(f"\n{col}:")
        value_counts = df[col].value_counts()
        print(value_counts)
        
        # Plot bar chart
        if len(value_counts) <= 20:  # Only plot if not too many categories
            plt.figure(figsize=(10, 6))
            value_counts.plot(kind='bar')
            plt.title(f'Distribution of {col}')
            plt.xlabel(col)
            plt.ylabel('Count')
            plt.xticks(rotation=45, ha='right')
            plt.tight_layout()
            plt.show()
else:
    print("[WARNING] Dataset not loaded.")
